# Notebook One
Technical areas included/ to include:
* Cleaning/ understanding the data
* Detailed data visualisation.
* Text blob, word cloud to see how descriptions of games are working.
* Machine learning processes. LMfit, linear regression.


- Who is your customer? 
  * Small video game company CEO or director of marketing/sales, deciding which direction their future projects should take. What genre, what platform, what area of the world to target it, how to name it, how to describe it, in general how to market so it sells well.
  
  
- What are they concerned with?
  * Increasing Revenues by increasing video game sales, by learning the correct areas to target their project.
  
  
- What would you like to find out for your customer?  
  * Finding out which platform of game there is a gap in the market for.
  * Finding out which genre of game ia likely to sell.
  * Finding out which area is growing and which area should be targeted. 
  * As a small company how impacted are the sales likely to be in comparison to a larger company. 
  * Finding out the impact game reviews have on sales.
  * Finding out which type of games are most likely to receive high reviews. 
  * Find out what types of words to describe games most influence the number of sales.
  * Find out which words are used to describe which genres are how influential they are in detailing the genre of game. 

# Loading in and cleaning the selected datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import requests as rq
import gzip
from wordcloud import WordCloud
import scipy as sp
import plotly
import plotly.plotly as py
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from scipy import stats
init_notebook_mode(connected=True) 
sns.set(color_codes=True)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew
from scipy.stats.stats import pearsonr
import matplotlib.pyplot as plt
from scipy.stats import skew
from scipy.stats.stats import pearsonr


%config InlineBackend.figure_format = 'retina' #set 'png' here when working on notebook
%matplotlib inline

%matplotlib inline
import warnings

warnings.filterwarnings("ignore")

In [ ]:
sales_df = pd.read_csv('../input/video-game-sales-with-ratings/Video_Games_Sales_as_at_22_Dec_2016.csv', na_values=['NA'])

I have chosen to select the games sales dataframe:
From looking at this, this is a good dataframe in terms of having alot to analyse in terms of figure not just in terms of sales but also in terms of it's release year and more. Much more detail than IGN or steam data.

In [ ]:
#Good dataframe, lots to analyse, lots of information and alot of games.
sales_df.head()

Seeing how the dataset is broken down, how many null values are there? Are they in the appropriate datatype, how many rows are there, should I drop any? etc...

In [ ]:
sales_df.info()

In [ ]:
sales_df.describe()

In [ ]:
sales_df.dtypes

# games_sales dataset
This dataset comes from: 'https://www.kaggle.com/gregorut/videogamesales' and is then built on at: 'https://www.kaggle.com/rush4ratio/video-game-sales-with-ratings' 
The original dataset is described as ' This dataset contains a list of video games with sales greater than 100,000 copies. It was generated by a scrape of vgchartz.com.' And then being built on is desribed as 'Motivated by Gregory Smith's web scrape of VGChartz Video Games Sales, this data set simply extends the number of variables with another web scrape from Metacritic. Unfortunately, there are missing observations as Metacritic only covers a subset of the platforms. Also, a game may not have all the observations of the additional variables discussed below. Complete cases are ~ 6,900'

Potential Limitations from above:
- From the describe function it is clear to see that there are quite alot of missing values within a number of the columns this will have to be addressed.
- There are clearly significant amounts of duplicate data this must be dealt with also.

The columns included are described as: 
- Name - The games name, changed to title and merged with previous data
- Platform - Platform of the games release (i.e. PC,PS4, etc.) from vgchartz.com.
- Year - Year of the game's release from vgchartz.com. 
- Genre - Genre of the game from vgchartz.com.
- Publisher - Publisher of the game, from vgchartz.com. This describes who publishes the game. There should be no issues with this data.
- NA_Sales - Sales in North America (in millions) from vgchartz.com, this should also bring no issues in relation to analysis.
- EU_Sales - Sales in Europe (in millions) from vgchartz.com, this should also bring no issues in relation to analysis.
- JP_Sales - Sales in Japan (in millions) from vgchartz.com, this should also bring no issues in relation to analysis.
- Other_Sales - Sales in the rest of the world (in millions) from vgchartz.com, this should also bring no issues in relation to analysis.
- Global_Sales - Total worldwide sales.(in millions)  from vgchartz.com, this should also bring no issues in relation to analysis.
- Critic_score - Aggregate score compiled by Metacritic staff, from metacritic.com, should be changed to out of 10 as that is how the rest of the score are marked.
- Critic_count - The number of critics used in coming up with the score from above, pulled from the website. 
- User_score - Score by Metacritic's subscribers, from the website.
- User_count - Number of users who gave the user_score, from the website. This may be dropped as it has little potential to be useful for analysis
- Developer - Party responsible for creating the game
- Rating - The ESRB ratings, all from the metacritic and vgchartz.com websites. 

Begin to clean the dataset, by dropping duplicates, filling or dropping columns with too many null rows, dropping useless rows and ensuring data is of the appropriate type.

In [ ]:
sales_df.drop_duplicates(subset=['Name', 'Platform'], inplace=True)

In [ ]:
sales_df.drop(['Critic_Count'] ,inplace =True, axis =1)
sales_df.drop(['User_Count'] ,inplace =True, axis =1)

Another issue is that user score is an object this needs to be changed, this needs addressed, so for future analysis it is all in the same format and can be visualised appropriately.

In [ ]:
sales_df['User_Score'].unique()

In [ ]:
sales_df['User_Score'].fillna(0,inplace=True)

In [ ]:
sales_df['User_Score'] = sales_df['User_Score'].replace('tbd', np.nan)

In [ ]:
sales_df['User_Score'].fillna(0,inplace=True)

In [ ]:
sales_df['User_Score'].replace(0, np.nan, inplace=True)

In [ ]:
sales_df.User_Score = sales_df.User_Score.astype(float)

In [ ]:
sales_df.info()

Due to the significant amount of missing data we need to deal with it appropriately, by filling it with relevant information or just dropping it to ensure it won't skew the data in future.

In [ ]:
sales_df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
total = sales_df.isnull().sum().sort_values(ascending=False)
percent = (sales_df.isnull().sum()/sales_df['Global_Sales'].count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
missing_column_sales= missing_data[missing_data['Total']>0].index
missing_column_sales

In [ ]:
sales_df[pd.isnull(sales_df['Name'])]

In [ ]:
sales_df.drop(659, inplace =True)

In [ ]:
sales_df[pd.isnull(sales_df['Name'])]

In [ ]:
sales_df['Publisher'].fillna('Unknown', inplace=True)

In [ ]:
sales_df.dropna(subset=['Year_of_Release'], inplace=True)

In [ ]:
sales_df['Rating'].fillna('Unknown', inplace=True)

In [ ]:
sales_df.drop(['Developer'] ,inplace =True, axis =1)

In [ ]:
sales_df.info()

In [ ]:
sales_df.head()

In [ ]:
sales_df.isnull().sum().sort_values(ascending=False).head(20)

For now I am going to leave some of the null values in the critic and user score as I will be mostly handling the other columns and the blank data later on will be dropped or filled with the mean of the column.
I have cleaned the dataset so that the blank columns are minimised, and from here it should make it easier to analyse.

Checking for outliers in terms of years released. For me, due to the fact this information is for a modern day game company, some of the much older data, is smaller and possibly much less relevant.
As a result of this I will filter some of the much more dated years out.

In [ ]:
sales_df.groupby(['Year_of_Release']).size()

In [ ]:
init_notebook_mode(connected=True)
year_count = sales_df.groupby('Year_of_Release', axis=0).count().reset_index()[['Year_of_Release','Name']]
year_count.Year_of_Release = year_count.Year_of_Release.astype('int')

In [ ]:
trace = go.Scatter(
    x = year_count.Year_of_Release,
    y = year_count.Name,
    mode = 'lines',
    name = 'lines'
    
)


layout = go.Layout(
    title='Release by Year',
    yaxis=dict(
        title='Count'
    ),
    xaxis=dict(
        title='Year'
    ),
    height=600, width=600
)

fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

* From this we can see and the sort by year we can see that were significantly less games from 1980 until the late ninties, as a result, I would say the games before 1996 or so would be considered outliers for a number of different reasons. Most significantly they were produced in a much less competitive market and it is important that these games may skew any findings as a director of sales would only be interested in the results from games in the not so distant past in my opinion. 

* Further to this I can see that despite the fact this data was produced in 2016 it includes data from 2017 onwards, these should also be dropped. 

In [ ]:
drop_year = [1980.0, 1981.0, 1982.0,1983.0,1984.0,1985.0,1986.0,1987.0,1988.0,1989.0, 1990.0, 1991.0,1992.0,1993.0, 1994.0, 1995.0 ,2017.0,2020.0]

In [ ]:
sales_df[~sales_df.Year_of_Release.isin(drop_year)]

This creates a new data frame only including games released from 1996 - 2016 
A better range of games discounting some outliers

In [ ]:
sales = sales_df[~sales_df.Year_of_Release.isin(drop_year)]

In [ ]:
sales.groupby(['Year_of_Release']).size()

In [ ]:
sales.head()

In [ ]:
sales.info()

In [ ]:
sales.dtypes

In [ ]:
sales.describe()

At this point the dataframe is clean and ready for visualisation and analysis

# Importing an external dataframe
'https://data.world/craigkelly/steam-game-data' - This is the link contains a significant data from games that are all sold on the platform STEAM. Although alot of the games here aren't PC games sold on steam their game reviews will generally be the same despite the platform. Same thing goes for the game price and a number of other useful factors I will be utilising in order to analyse much of the data in greater detail.

In [ ]:
steam_data = pd.read_csv("../input/steam-data-info/games-features.csv")

From the documentation of this each column means the following: QueryID - (Integer) The original ID in idlist.csv

- ResponseID - (Integer) The ID returned in the Steam response (should equal QueryID)

- QueryName - (Text) The original name in idlist.csv

- ResponseName - (Text) The name returned in the Steam response (should equal QueryName)

- ReleaseDate - (Text) Appears to the be the initial release date for the game

- RequiredAge - (Integer) list named required_age in JSON

- DemoCount - (TextualCount) list named demos in JSON

- DeveloperCount - (TextualCount) list named developers in JSON

- DLCCount - (TextualCount) list named dlc in JSON

- Metacritic - (Integer) numeric score from metacritic object in JSON

- MovieCount - (TextualCount) list named movies in JSON (used object id for unique count)

- PackageCount - (TextualCount) list named packages in JSON

- RecommendationCount - (Integer) from recommendations.total in JSON

- PublisherCount - (TextualCount) list named publishers in JSON

- ScreenshotCount - (TextualCount) list named screenshots in JSON

- AchievementCount - (Integer) achievements.total in JSON

- AchievementHighlightedCount - (TextualCount) for achievements.highlighted in JSON

- ControllerSupport - (Boolean) True if controller_support was full

- IsFree - (Boolean) is_free in JSON

- FreeVerAvail - (Boolean) True if is_free_license is True in package_groups list

- PurchaseAvail - (Boolean) True if price_in_cents_with_discount greater than 0 in package_groups list

- SubscriptionAvail - (Boolean) True if is_recurring_subscription is True in package_groups

- PlatformWindows - (Boolean) True if platforms.windows is True

- PlatformLinux - (Boolean) True if platforms.linux is True

- PlatformMac - (Boolean) True if platforms.mac is True

- PCReqsHaveMin - (Boolean) True if pc_requirements.minimum is non-empty string

- PCReqsHaveRec - (Boolean) True if pc_requirements.recommended is non-empty string

- LinuxReqsHaveMin - (Boolean) True if linux_requirements.minimum is non-empty string

- LinuxReqsHaveRec - (Boolean) True if linux_requirements.recommended is non-empty string

- MacReqsHaveMin - (Boolean) True if mac_requirements.minimum is non-empty string

- MacReqsHaveRec - (Boolean) True if mac_requirements.recommended is non-empty string

- CategorySinglePlayer - (Boolean) True if for any i, categories[i].description is "single-player"

- CategoryMultiplayer - (Boolean) True if for any i, categories[i].description is one of: "cross-platform multiplayer", "local multi-player", "multi-player", "online multi-player", "shared/split screen"

- CategoryCoop - (Boolean) True if for any i, categories[i].description is one of: "co-op", "local co-op", "online co-op"

- CategoryMMO - (Boolean) True if for any i, categories[i].description is "mmo"

- CategoryInAppPurchase - (Boolean) True if for any i, categories[i].description is "in-app purchases"

- CategoryIncludeSrcSDK - (Boolean) True if for any i, categories[i].description is "includes source sdk"

- CategoryIncludeLevelEditor - (Boolean) True if for any i, categories[i].description is "includes level editor"

- CategoryVRSupport - (Boolean) True if for any i, categories[i].description is "vr support"

- GenreIsNonGame - (Boolean) True if for any i, genres[i].description is one of: "utilities", "design & illustration", "animation & modeling", "software training", "education", "audio production", "video production", "web publishing", "photo editing", "accounting"

- GenreIsIndie - (Boolean) True if for any i, genres[i].description is "indie"

- GenreIsAction - (Boolean) True if for any i, genres[i].description is "action"

- GenreIsAdventure - (Boolean) True if for any i, genres[i].description is "adventure"

- GenreIsCasual - (Boolean) True if for any i, genres[i].description is "casual"

- GenreIsStrategy - (Boolean) True if for any i, genres[i].description is "strategy"

- GenreIsRPG - (Boolean) True if for any i, genres[i].description is "rpg"

- GenreIsSimulation - (Boolean) True if for any i, genres[i].description is "simulation"

- GenreIsEarlyAccess - (Boolean) True if for any i, genres[i].description is "early access"

- GenreIsFreeToPlay - (Boolean) True if for any i, genres[i].description is "free to play"

- GenreIsSports - (Boolean) True if for any i, genres[i].description is "sports"

- GenreIsRacing - (Boolean) True if for any i, genres[i].description is "racing"

- GenreIsMassivelyMultiplayer - (Boolean) True if for any i, genres[i].description is "massively multiplayer"

- PriceCurrency - (Text) price_overview.currency in JSON

- PriceInitial - (Float) price_overview.initial in JSON, divided by 100.0 to converts cents to currency

- PriceFinal - (Float) price_overview.final in JSON, divided by 100.0 to converts cents to currency

- SteamSpyOwners - (steamspy.com) total owners, which includes free weekend trials and other possibly spurious numbers.

- SteamSpyOwnersVariance - (steamspy.com) total owners, which includes free weekend trials and other possibly spurious numbers. Note that this is not technically variance: according to steamspy.com, "the real number... lies somewhere on... [value +/- variance]"

- SteamSpyPlayersEstimate - (steamspy.com) best estimate of total number of people who have played the game since March 2009

- SteamSpyPlayersVariance - (steamspy.com) errors bounds on SteamSpyPlayersEstimate. Note that this is not technically variance: according to steamspy.com, "the real number... lies somewhere on... [value +/- variance]"

- SupportEmail - (Textual) support_info.email in JSON

- SupportURL - (Textual) support_info.url in JSON

- AboutText - (Textual) about_the_game in JSON

- Background - (Textual) background in JSON

- ShortDescrip - (Textual) short_description in JSON

- DetailedDescrip - (Textual) detailed_description in JSON

- DRMNotice - (Textual) drm_notice in JSON

- ExtUserAcctNotice - (Textual) ext_user_account_notice in JSON

- HeaderImage - (Textual) header_image in JSON

- LegalNotice - (Textual) legal_notice in JSON

- Reviews - (Textual) reviews in JSON

- SupportedLanguages - (Textual) supported_languages in JSON

- Website - (Textual) website in JSON

- PCMinReqsText - (Textual) text of pc_requirements.minimum

- PCRecReqsText - (Textual) text of pc_requirements.recommended

- LinuxMinReqsText - (Textual) text of linux_requirements.minimum

- LinuxRecReqsText - (Textual) text of linux_requirements.recommended

- MacMinReqsText - (Textual) text of mac_requirements.minimum

- MacRecReqsText - (Textual) text of mac_requirements.recommended

As you can see this contains alot of useful information, however it also contains significant information we either already have or information that is completely useless as a result I will drop the rows I can not use as below.
The only rows I will keep are as follows 
- QueryName
- RequiredAge - This has many blanks columns and needs resolved
- PriceFinal - This will be useful to analyse what sales are like depending on price
- AboutText - Useful for Textblob and sentiment analysis
- ShortDescrip- Useful for Textblob and sentiment analysis
- Detailed Descrip - Useful for Textblob and sentiment analysis
- Reviews -  Useful for Textblob and sentiment analysis

This must be cleaned the same way the the game sales one was done.

In [ ]:
steam_data.head()

As stated above and seen above, there is useful information in this dataset, however there is also significant amounts of useless data, this will be dropped below.

In [ ]:
steam_data.drop(['RequiredAge' ,'ResponseID','QueryID', 'ResponseName','DemoCount','DeveloperCount','DLCCount','MovieCount','PackageCount',
 'PublisherCount','ScreenshotCount','AchievementCount','AchievementHighlightedCount','ControllerSupport',
 'IsFree', 'FreeVerAvail', 'PurchaseAvail',
 'SubscriptionAvail','PlatformWindows','PlatformLinux','PlatformMac','PCReqsHaveMin','PCReqsHaveRec','LinuxReqsHaveMin','LinuxReqsHaveRec','MacReqsHaveMin','MacReqsHaveRec','CategorySinglePlayer',
                  'CategoryMultiplayer','CategoryCoop','CategoryMMO','CategoryInAppPurchase','CategoryIncludeSrcSDK','CategoryIncludeLevelEditor','CategoryVRSupport','GenreIsNonGame','GenreIsIndie','GenreIsAction',
                  'GenreIsAdventure','GenreIsCasual','GenreIsStrategy',
                  'GenreIsRPG','GenreIsSimulation','GenreIsEarlyAccess','GenreIsFreeToPlay',
                  'GenreIsSports','GenreIsRacing','GenreIsMassivelyMultiplayer',
 'SupportEmail', 'SupportURL','DRMNotice','ExtUserAcctNotice','HeaderImage','LegalNotice', 'SupportedLanguages','Website','PCMinReqsText','PCRecReqsText','LinuxMinReqsText','LinuxRecReqsText','MacMinReqsText','MacRecReqsText','Background','PriceCurrency', 'SteamSpyPlayersVariance', 'SteamSpyOwners' , 'SteamSpyOwners', 'Metacritic', 'SteamSpyOwnersVariance','SteamSpyPlayersEstimate', 'RecommendationCount', 'ReleaseDate', 'PriceInitial'], inplace=True, axis = 1)

In [ ]:
steam_data.head()

In [ ]:
steam_data.info()

At this point so the dataset is cleaned up but we have to deal will the blank data, which as you can see above there is plenty of and it seems there is more than can be seen straight away, as a result we have to changed the form of some columns and make sure all Nan values are easily found. 

In [ ]:
steam_data[pd.isnull(steam_data['QueryName'])]

In [ ]:
steam_data.drop(steam_data.index[269], inplace =True )

In [ ]:
steam_data.replace(" ", np.NaN, inplace=True)

In [ ]:
steam_data.head()

In [ ]:
steam_data.dropna(subset=['DetailedDescrip'], inplace = True)

In [ ]:
steam_data.info()

In [ ]:
steam_data.head()

In [ ]:
#Change this column name to 'Name' so 
#it can be joined with the sales df easier if necessary
steam_data=steam_data.rename(columns = {'QueryName' : 'Name'})

# Merging the sales and the steam dataset.
Despite not being potentially as good as the amazon data I have failed to successfully scrape the information from it.

As a result I will work from the game sales and steam reviews dataframes. 

These will be merged together with the sales dataframe being the main focus due to the fact it has more useful and analysable data. 

In [ ]:
games_df = sales.merge(steam_data, how='left', on='Name')

In [ ]:
games_df.info()

In [ ]:
games_df.head()

# Platform analysis & visualistion

For my visualisation and my interpretation of the data, for the sake of my client we are going to work to provide analysis from the start of the dataset but also shrink the dataset down and analyse trends the start of the release of the next generation consoles as that is the most relevant data. This is important as redundant consoles potentially aren't worth analysing unless there is an overall picture provided. I.e. the N64 etc.
Credit to: 
 - https://www.kaggle.com/jiehu15/eda-by-plotly-on-game-sales-dataset

In [ ]:
games_df.Platform = games_df.Platform.astype('category')
games_df.Platform.describe()

In [ ]:
games_df.info()

In [ ]:
relevant_years = [2014.0, 2015.0, 2016.0]

In [ ]:
relevant_games = games_df[games_df.Year_of_Release.isin(relevant_years)]

This shows the value of breaking the dataset down into more recent data. As PS2 has the most releases, the PS4 in the past few years has much more than the PS2 as it is outdated. 

In [ ]:
relevant_games.Platform = games_df.Platform.astype('category')
relevant_games.Platform.describe()

In [ ]:
relevant_games['Year_of_Release'].unique()

In [ ]:
relevant_games.head()

In this part of the notebook I visualise the difference in releases and sales globally based on the platform which the game is released on. 

In [ ]:
platform_count = games_df.groupby('Platform', axis=0).count().reset_index()[['Platform','Name']].sort_values(by = "Name", ascending=True)

In [ ]:
layout = go.Layout(
    title='Total Release by Platforms',
    yaxis=dict(
        title='Platform'
    ),
    xaxis=dict(
        title='Count'
    ),
    height=600, width=600
)

trace = go.Bar(
            x=platform_count.Name,
            y=platform_count.Platform,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig, show_link=False)

In [ ]:
platGenre = pd.crosstab(games_df.Platform,games_df.Genre)
platGenreTotal = platGenre.sum(axis=1).sort_values(ascending = False)
plt.figure(figsize=(8,6))
sns.barplot(y = platGenreTotal.index, x = platGenreTotal.values, orient='h')
plt.ylabel = "Platform"
plt.xlabel = "The amount of games"
plt.show()

This is the reason I am going to do further analysis in that I am going to split it up into more modern years. As I would suggest if I was going to release a game today that you should not focus on the PS2. Simply it has been out the longest.

In [ ]:
platform_count = relevant_games.groupby('Platform', axis=0).count().reset_index()[['Platform','Name']].sort_values(by = "Name", ascending=True)

In [ ]:
layout = go.Layout(
    title='Total Release by Platforms, 2013-2016',
    yaxis=dict(
        title='Platform'
    ),
    xaxis=dict(
        title='Count'
    ),
    height=600, width=600
)

trace = go.Bar(
            x=platform_count.Name,
            y=platform_count.Platform,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig, show_link=False)

In [ ]:
platGenre = pd.crosstab(relevant_games.Platform,relevant_games.Genre)
platGenreTotal = platGenre.sum(axis=1).sort_values(ascending = False)
plt.figure(figsize=(8,6))
sns.barplot(y = platGenreTotal.index, x = platGenreTotal.values, orient='h')
plt.ylabel = "Platform"
plt.xlabel = "The amount of games"
plt.show()

Surprisingly to me, even more games have been released onto the Playstation Vita, from this it seems like the Playstations are dominating on releases and if a developer were to develop a new game I would suggest that a focus should be developing on the PS4.

In [ ]:
table_count = pd.pivot_table(relevant_games,values=['Global_Sales'],index=['Year_of_Release'],columns=['Platform'],aggfunc='count',margins=False)

plt.figure(figsize=(19,16))
sns.heatmap(table_count['Global_Sales'],linewidths=.5,annot=True,fmt='2.0f',vmin=0)
plt.title('Game Releases')

Based on the above visualisation I can say: 
 - PS4 and xbox releases are the most common and are the main market for releases
 - Past that the previous generation consoles still receive new games, but less by the year.
 - Furthmore, WiiU and Nintendo products continued to be released at a slower rate but they still have a strong market. 
 - PC is a smaller market but definitely not deteriorating, growing the past few years.

In [ ]:
sns.factorplot(x="Platform", y="Global_Sales", data=relevant_games, size= 4, aspect =2)

In [ ]:
sales_by_platform = games_df.groupby(['Platform','Name'], axis = 0).sum().reset_index()[['Platform','Name','Global_Sales']]

In [ ]:
import random
from numpy import * 
platforms = sales_by_platform.Platform.unique()
traces = []
c = ['hsl('+str(h)+',50%'+',50%)' for h in linspace(0, 360, len(platforms))]

for i in range(len(platforms)):
    platform= platforms[i]
    df_platform = sales_by_platform[sales_by_platform.Platform == platform]
    trace = go.Box(
        y=np.array(df_platform.Global_Sales),
        name=platform,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by platform (Alot of outliers) ',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)


In [ ]:
# The outliers consist of:
games_df.groupby(['Platform','Name'], axis = 0).\
         sum()[['Global_Sales']].\
         sort_values(by="Global_Sales", ascending = False).\
         reset_index()[:20]

For the following, it is obvious that there are some games which are significantly more successful than can be useful for the purposes of analysis, particularly from the perspective of a smaller company as these games usually have large company backing and go viral. 
As a result I will analyse the platform sales taking out the outliers.

In [ ]:
PERCENTAGE = 0.99
traces = []

for i in range(len(platforms)):
    platform = platforms[i]
    df_platform = sales_by_platform[sales_by_platform.Platform == platform]
    df_platform = df_platform[df_platform.Global_Sales > df_platform.Global_Sales.quantile(PERCENTAGE)]
    
    trace = go.Box(
        y=np.array(df_platform.Global_Sales),
        name=platform,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by platform (TOP 1% games)',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)

From this visualisation we can see many of the games which have sold amounts which could be considered anomolies are on platforms which have been around for a long time, i.e. previous generation consoles, and these tend to have a weirder distrubution. For our purpose this is less useful for any analysis due to the skewed distribution. 

In [ ]:
PERCENTAGE = 0.95
traces = []

for i in range(len(platforms)):
    platform = platforms[i]
    df_platform = sales_by_platform[sales_by_platform.Platform == platform]
    df_platform = df_platform[df_platform.Global_Sales < df_platform.Global_Sales.quantile(PERCENTAGE)]
    
    trace = go.Box(
        y=np.array(df_platform.Global_Sales),
        name=platform,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by platform (significant outliers dropped)',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)

With the outliers removed we can see that even considering how long the PS4, and Xbox one have been out they are selling similarly well to their older rivals, with the data less skewed, we can see that ntomaling most consoles are selling around the same within their normal distrubtion but the PS4 have a better number of high sellers than most of the new consoles. 

In [ ]:
table = games_df.pivot_table('Global_Sales', index='Platform', columns='Year_of_Release', aggfunc='sum')
platforms = table.idxmax()
annualsales = table.max()
years = table.columns.astype(int)
data = pd.concat([platforms, annualsales], axis=1)
data.columns = ['Platform', 'Global Sales']

plt.figure(figsize=(12,8))
ax = sns.pointplot(y = 'Global Sales', x = years, hue='Platform', data=data, size=15)
ax.set_xlabel(xlabel='Year', fontsize=16)
ax.set_ylabel(ylabel='Global Sales Per Year', fontsize=16)
ax.set_title(label='Highest Total Platform Revenue in $ Millions Per Year', fontsize=20)
ax.set_xticklabels(labels = years, fontsize=12, rotation=50)
plt.show();

The above diagram does a great job at demonstrating that there is generally an era where a games console will be the leading console, which has went from Ps, to Wii, 360, PS3 and finally being our current generation of the PS4, backing up the point I would suggest the PS4 is the current best platform to focus on. 

The below code focuses on the impact of the platform of the game and how it sells in different geographical locations.

In [ ]:
platforms = np.sort(sales.Platform.unique())[::-1]

def get_traces2(sales, region):
    regional_df = sales.groupby(['Platform','Year_of_Release'], axis=0).sum().reset_index()[['Platform','Year_of_Release', region]]
    years = list(range(1996,2018))
    
    temp_dict = {}
    for platform in platforms:
        temp_dict[platform] = {}
        for year in years:
            try:
                temp_value = round(np.array(regional_df[(regional_df.Platform == platform) & 
                                   (regional_df.Year_of_Release == year)][region])[0],2)
            except:
                temp_value = 0
            temp_dict[platform][year] = temp_value
    
    traces = []
    for platform in platforms:
        trace = go.Bar(
            x = years,
            y = list(temp_dict[platform].values()),
            name=platform
        )
        traces.append(trace)
    
    return traces

In [ ]:
print(platforms)

In [ ]:
data = get_traces2(sales, 'Global_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Global',
        xaxis=dict(
            title='Year_of_Release'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

The above diagram backs up the point made above, the current console of importance is the PS4 selling 70m copies in the midway point of 2016, which is impressive in comparison to it's rivals. 
In 2015 and 2016 the top consoles globally consist of: 
1. PS4
2. XBox One
3. 3DS (Surprisingly) 
4. WiiU
5. PC

In [ ]:
data = get_traces2(games_df, 'NA_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in North America',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

North American Consoles to focus on: 
1. PS4
2. Xbox One
3. 3DS
4. WiiU
5. PC

In [ ]:
data = get_traces2(games_df, 'EU_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Europe',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

EU top recent platforms: 
1. PS4 (Massively popular) 
2. Xbox one (Much less than the PS4) 
3. PC
4. 3DS
5. WiiU 

In [ ]:
data = get_traces2(games_df, 'JP_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Japan',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

Japan top platforms, dramatically different from global trends: 
1. 3DS is massive
2. PS4
3. PSVita (anomoly for JP) 
4. WiiU
5. PS3

Surprising as literally no PC games are played in JP and pretty much all Microsoft is none existent. Proving that what region you aim your game at is very important.

In [ ]:
def get_percent_traces2(df, region):
    temp_df = df.groupby(['Year_of_Release','Platform'], axis=0).sum()[[region]]
    df_pcts = temp_df.groupby(level=0).apply(lambda x: 100 * x / float(x.sum()))
    df_pcts = df_pcts.reset_index()
    regional_df = df_pcts[df_pcts.Year_of_Release < 2017] 
    
    years = list(range(1996,2018))
    
    temp_dict = {}
    for platform in platforms:
        temp_dict[platform] = {}
        for year in years:
            try:
                temp_value = round(np.array(regional_df[(regional_df.Platform == platform) & 
                                   (regional_df.Year_of_Release == year)][region])[0],2)
            except:
                temp_value = 0
            temp_dict[platform][year] = temp_value
    
    
    traces = []
    for platform in platforms:
        trace = go.Bar(
            x = years,
            y = list(temp_dict[platform].values()),
            name=platform
        )
        traces.append(trace)
    
    return traces

In [ ]:
data = get_percent_traces2(games_df, 'Global_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales Percentage of Platforms over Years in Global',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

Once again, this proves on a global scale, the PS4 (Over 50%!) is the way to go if hoping for a decent sales figure, followed by the XBox one (20%) then the 3DS, smaller PC & WiiU and a few sales on older platforms. 
From here we can definitely see that PS4 is the platform for the future to focus on and as a result I would recommend any publisher focuses on it to maximise sales. 

From here we need further visualisation of the impact of the publisher and genre of the game.

In [ ]:
plt.subplots(figsize=(15,15))
max_platforms=games_df.groupby('Platform')['Platform'].count()
max_platforms=max_platforms[max_platforms.values>1]
max_platforms.sort_values(ascending=True,inplace=True)
mean_games=games_df[games_df['Platform'].isin(max_platforms.index)]
abc=mean_games.groupby(['Year_of_Release','Platform'])['Critic_Score'].mean().reset_index()
abc=abc.pivot('Year_of_Release','Platform','Critic_Score')
sns.heatmap(abc,annot=True,cmap='RdYlGn',linewidths=0.4)
plt.title('Average Score By Platforms')
plt.show()

This is useful as it shows us the remaining relevant platforms: PS4, PC, 3DS, XBox One, WiiU and PSV.
From this we can clearly see that the platforms remaining do not cause a significant difference to the score provided by critic. Which makes sense, so from the point of view the platform isn't as relevant.

# Publisher analysis and visualisation

In [ ]:
relevant_games["Year_of_Release"].unique()

In [ ]:
publisher_sales = relevant_games.groupby('Publisher', axis=0).sum().reset_index()[['Publisher','Global_Sales']].sort_values(by = "Global_Sales", ascending=True)
publisher_sales = publisher_sales.tail(n=30)

layout = go.Layout(
    title='Sales by Publisher (Top 30) 2014-2016' ,

    xaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=300,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)

trace = go.Bar(
            x=publisher_sales.Global_Sales,
            y=publisher_sales.Publisher,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

In [ ]:
most_pub = relevant_games.groupby('Publisher').Global_Sales.sum()
most_pub.sort_values(ascending=False)[:20]

table_publisher = pd.pivot_table(relevant_games[relevant_games.Publisher.isin(most_pub.sort_values(ascending=False)[:20].index)],values=['Global_Sales'],index=['Year_of_Release'],columns=['Publisher'],aggfunc='sum',margins=False)


plt.figure(figsize=(19,16))
sns.heatmap(table_publisher['Global_Sales'],linewidths=.5,annot=True,vmin=0.01,cmap='PuBu')
plt.title('Sum Publisher Global_sales of games')

From the above diagrams we can see that the bigger publishers i.e. EA, Activision, Nintendo etc are the highest sellers in the years of 2014-2016. 

It is important to note however they put out the most amount of games and as a result these may not accurately reflect how many sales per game they have as a result I will visualise this below.

In [ ]:
new_df = relevant_games
new_df['Game_Count'] = 1
new_df = new_df.groupby(['Publisher']).sum().reset_index()[['Publisher', 'Global_Sales','Game_Count']]
new_df['Revenue_per_game'] = new_df.Global_Sales/new_df.Game_Count

new_df = new_df.sort_values(by = "Revenue_per_game", ascending=True).\
                            tail(n=30)
layout = go.Layout(
    title='Revenue_per_game by Publisher (Top 30) 2014-2016',

    xaxis=dict(
        title='Revenue_per_game (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=250,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)

trace = go.Bar(
            x=new_df.Revenue_per_game,
            y=new_df.Publisher,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

When it is broken down further, we can see that per game the larger companies are still some of the best selling but there is more variety and some smaller publishers are in and around the big ones unlike the graph above.
 
As a result of this, I would suggest even though having a big publisher put the game out is an advantage an indie company can sell well as suggested by the above 'Hello Games'.

From here it is important that I am able to visualise the impact that the Genre of the game has on the sales.

# Genre analysis and visualisation

In [ ]:
genre_count = games_df.groupby('Genre', axis=0).count().reset_index()[['Genre','Name']].sort_values(by = "Name", ascending=True)
layout = go.Layout(
    title='Releases by Genre, 1996-2016',
   
    xaxis=dict(
        title='Releases'
    ),
    height=400, width=600
)

trace = go.Bar(
            x=genre_count.Name,
            y=genre_count.Genre,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

In [ ]:
genre_count = relevant_games.groupby('Genre', axis=0).count().reset_index()[['Genre','Name']].sort_values(by = "Name", ascending=True)
layout = go.Layout(
    title='Releases by Genre, 2014-2016',
   
    xaxis=dict(
        title='Releases'
    ),
    height=400, width=600
)

trace = go.Bar(
            x=genre_count.Name,
            y=genre_count.Genre,
            orientation = 'h'
        )


fig = go.Figure(data=[trace], layout=layout)
iplot(fig)

As we can see here, since 1996 the most publisher game is from the Action genre. 
More important however, is the fact that from 2014-2016 the % of action releases grew by a large amount and they are now a main part of the market. This suggests: 
- Action games are the most popular releases but; 
- Action games are also the most crowded market.

In [ ]:
plt.subplots(figsize=(15,15))
max_genres=games_df.groupby('Genre')['Genre'].count()
max_genres=max_genres[max_genres.values>1]
max_genres.sort_values(ascending=True,inplace=True)
mean_games=games_df[games_df['Genre'].isin(max_genres.index)]
abc=mean_games.groupby(['Year_of_Release','Genre'])['Critic_Score'].mean().reset_index()
abc=abc.pivot('Year_of_Release','Genre','Critic_Score')
sns.heatmap(abc,annot=True,cmap='RdYlGn',linewidths=0.4)
plt.title('Average Score By Genre"s')
plt.show()

From a ctritic_score point of the genre is less important, as the scores vary greatly the past few yasrs Action games have scored consistently whilst Strategy and Platform games clearly depend on the games released. 
From this I would draw the conclusion that: 
- Hit games in terms of critic score come in genres such as role playing, Strategy and platform games. 
- Games like action, shooter and fighting tending to get more consistently decent scores and most seem to get around the 70 mark. 

In [ ]:
table_count = pd.pivot_table(relevant_games,values=['Global_Sales'],index=['Year_of_Release'],columns=['Genre'],aggfunc='count',margins=False)

plt.figure(figsize=(14,10))
sns.heatmap(table_count['Global_Sales'],linewidths=.5,annot=True,fmt='2.0f',vmin=0)
plt.title('Count of games')

As we can see from this the highest selling Genre is action games, this is really the only constant the other genres are usually less consistent depending on what is released that year but the other games which have a a large market are: 
1. Adventure
2. Role Playing
3. shooter 
4. sports
These are all genres alongside action that have a large market and are bought alot. 

From here we should look into whether action games relies on it's high selling games or whether on average it is the best genre to target, i.e. whether it's normal distribution is good or whether this genre relies on games going 'viral'. 

In [ ]:
sales_by_genre = relevant_games.groupby(['Genre','Name'], axis = 0).sum().reset_index()[['Genre','Name','Global_Sales']]

In [ ]:
import random
from numpy import * 
genres = sales_by_genre.Genre.unique()
traces = []
c = ['hsl('+str(h)+',50%'+',50%)' for h in linspace(0, 360, len(genres))]

for i in range(len(genres)):
    genre = genres[i]
    df_genre = sales_by_genre[sales_by_genre.Genre == genre]
    trace = go.Box(
        y=np.array(df_genre.Global_Sales),
        name=genre,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by Genre (A lot of outliers)',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)

As we can see here, none of the genres really have a great normal distribution, there are significant amounts of outliers skewing our visualisation, in order to see which games on average sell the best, we should analyse without the outliers. 

However, from this I can draw that action games DO rely on the genre's big sellers, i.e. COD and other big sellers they don't have the consistently high sellers which other genres do as we can see their normal distribution is no where near as good as Platform, Racing, Shooter and Sports. Meaning despite it being largely the best selling genre it may not be important to target games to it unless you expect it to catch on hugely. 

In [ ]:
# After delete outlier

PERCENTAGE = 0.95
traces = []

for i in range(len(genres)):
    genre = genres[i]
    df_genre = sales_by_genre[sales_by_genre.Genre == genre]
    df_genre = df_genre[df_genre.Global_Sales < df_genre.Global_Sales.quantile(PERCENTAGE)]
    
    trace = go.Box(
        y=np.array(df_genre.Global_Sales),
        name=genre,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by Genre (TOP 1% removed)',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)

This backs up what was said before, without action games huge outliers, the sales aren't as good as the normal distribution of shooters, platformers, sports, racing, fighting and role playing games. 
From this I would suggest Focusing releases on action games would not be recommened in a possibly over crowded and household name reliant genre. 

In [ ]:
PERCENTAGE = 0.99
traces = []

for i in range(len(genres)):
    genre = genres[i]
    df_genre = sales_by_genre[sales_by_genre.Genre == genre]
    df_genre = df_genre[df_genre.Global_Sales > df_genre.Global_Sales.quantile(PERCENTAGE)]
    
    trace = go.Box(
        y=np.array(df_genre.Global_Sales),
        name=genre,
        boxmean=True,
        marker={'color': c[i]}
    )
    
    traces.append(trace)

layout = go.Layout(
    title='Sales by Genre (TOP 1%)',
    showlegend=False,
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    height=700, width=700,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)
    

fig = go.Figure(data=traces, layout=layout)
iplot(fig)

This again backs up what is said above, the most common outliers are in the top 1%, it is obvious that without these the genre isn't as dominant in the market as it seems. 

In [ ]:
sns.violinplot(x="Global_Sales", y="Genre", data=relevant_games, size=8)
#sales per genre in the timeframe 2014-2016

Again we can see that other games have a better or as good spread of game sales as action games, meaning it is not the dominant game in the market anymore. 

In [ ]:
# Get list of unique genres
genres = np.sort(games_df.Genre.unique())[::-1]

def get_traces(games_df, region):
    regional_df = games_df.groupby(['Genre','Year_of_Release'], axis=0).sum().reset_index()[['Genre','Year_of_Release', region]]
    years = list(range(1996,2018))
    
    temp_dict = {}
    for genre in genres:
        temp_dict[genre] = {}
        for year in years:
            try:
                temp_value = round(np.array(regional_df[(regional_df.Genre == genre) & 
                                   (regional_df.Year_of_Release == year)][region])[0],2)
            except:
                temp_value = 0
            temp_dict[genre][year] = temp_value
    
    traces = []
    for genre in genres:
        trace = go.Bar(
            x = years,
            y = list(temp_dict[genre].values()),
            name=genre
        )
        traces.append(trace)
    
    return traces

In [ ]:
data = get_traces(games_df, 'Global_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Global',
        xaxis=dict(
            title='Year_of_Release'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=800, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

From this and the previous visualisations I can conclude that on a global scale, producers should focus on: 
1. Shooter games
2. Sports games
3. Role Playing
4. Action Games
5. Adventure

In [ ]:
data = get_traces(games_df, 'NA_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in North America',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=700, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

In [ ]:
data = get_traces(games_df, 'EU_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Europe',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=700, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

As the two main markets and sellers of global regions, the two EU and NA sales figures tend to follow the same pattern as the global sales patterns. Which is the following genres are most popular: 
1. Shooter games
2. Sports games
3. Role Playing
4. Action Games
5. Adventure

In [ ]:
data = get_traces(games_df, 'JP_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Japan',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=700, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

From this we can see again that the Japanese market doesn't follow the same patterns as the global trends, if the market of Japan were to be the developers target I would change my recommended course of action, and focus on the following genres: 
1. Role playing games
2. Action games
3. Fighting games

The rest of the genres have a very small market and as a result if it's a game outside of this genre perhaps JP isn't the idea target market. 

In [ ]:
data = get_traces(games_df, 'Other_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales change in Other regions',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=700, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

This follows the same patterns as the global sales statistics

In [ ]:
def get_percent_traces(df, region):
    temp_df = df.groupby(['Year_of_Release','Genre'], axis=0).sum()[[region]]
    df_pcts = temp_df.groupby(level=0).apply(lambda x: 100 * x / float(x.sum()))
    df_pcts = df_pcts.reset_index()
    regional_df = df_pcts[df_pcts.Year_of_Release < 2017] 
    
    years = list(range(1996,2018))
    
    temp_dict = {}
    for genre in genres:
        temp_dict[genre] = {}
        for year in years:
            try:
                temp_value = round(np.array(regional_df[(regional_df.Genre == genre) & 
                                   (regional_df.Year_of_Release == year)][region])[0],2)
            except:
                temp_value = 0
            temp_dict[genre][year] = temp_value
    
    
    traces = []
    for genre in genres:
        trace = go.Bar(
            x = years,
            y = list(temp_dict[genre].values()),
            name=genre
        )
        traces.append(trace)
    
    return traces

In [ ]:
data = get_percent_traces(games_df, 'Global_Sales')
layout = go.Layout(
        barmode='stack',
        title = 'Sales Percentage of Genres over Years in Global',
        xaxis=dict(
            title='Year'
        ),
        yaxis=dict(
            title='Sales (in Millions)'
        ),
    
        height=700, width=800,
        margin=go.Margin(
            l=100,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )
fig = go.Figure(data=data, layout=layout)
iplot(fig)

This gives us a great visualisation of the market share, from this I can state that the main genres to focus on in order are: 
1. Shooter
2. Sports 
3. Action
4. Role Playing
5. The other genres...

# Year by year, regional analysis

In [ ]:
sales_by_year = games_df.groupby('Year_of_Release', axis=0).sum().reset_index()[['Year_of_Release','NA_Sales','EU_Sales','JP_Sales','Other_Sales','Global_Sales']]
sales_by_year.Year = sales_by_year.Year_of_Release.astype('int')
sales_by_year.head()

# Regional analysis and visualisation

In [ ]:
trace_Global = go.Scatter(
    x = sales_by_year.Year,
    y = sales_by_year.Global_Sales,
    mode = 'none',
    name = 'Global_Sales',
    fill='tonexty',
)

trace_NA = go.Scatter(
    x = sales_by_year.Year,
    y = sales_by_year.NA_Sales,
    mode = 'none',
    fill='tonexty',
    name = 'NA_Sales'
)

trace_EU = go.Scatter(
    x = sales_by_year.Year,
    y = sales_by_year.EU_Sales,
    mode = 'none',
    fill='tonexty',
    name = 'EU_Sales'
)

trace_JP = go.Scatter(
    x = sales_by_year.Year,
    y = sales_by_year.JP_Sales,
    mode = 'none',
    fill='tonexty',
    name = 'JP_Sales'
)

trace_Other = go.Scatter(
    x = sales_by_year.Year,
    y = sales_by_year.Other_Sales,
    mode = 'none',
    fill='tozeroy',
    name = 'Other_Sales'
)



layout = go.Layout(
    title='Sales by Region',

    xaxis=dict(
        title='Year'
    ),
    yaxis=dict(
        title='Sales (in Millions)'
    ),
    
    height=600, width=800,
    margin=go.Margin(
        l=100,
        r=50,
        b=100,
        t=100,
        pad=4
    )
)


fig = go.Figure(data=[trace_Other, trace_JP, trace_EU, trace_NA, trace_Global], layout=layout)
iplot(fig)

From this I can draw the conclusion that the most influential regions are EU and NA in terms of sales. 
However, the sales figures from NA aren't as huge as they used to be, important to note is the raising importance of the EU sales, however due to their similar taste in games I would suggest that on a global scale focussing on their tastes is important to maximise sales.

As an overall summary from the above visualisations, I would suggest to a games marketing executive that, all aspects of a game i.e. genre, platform and publisher will affect the popularity and sales of the game. 
Further to this, consideration needs to be given to the area which you are targetting. I would summarise in the following manner: 

EU/NA/Other area: 
- platform recommendation: PS4, Xbox One, 3DS, WiiU, PC
- Genre recommendation: Shooter, Sports games, Role Playing, Action Games, Adventure

JP area:
- platform recommendation: 3DS, PS4, WiiU 
- Genre recommendation: Role playing games, Action games, Fighting games


From this I would suggest that despite being a handicap as a small publisher, there is still a good chance that games released although not as many that the games will sell well even with large company competition.

# - Using the steam dataframe to utilise with wordcloud which words usually describe genres and best sellers. Etc.

- Credit to: https://www.kaggle.com/rosado/sentiment-analysis-text-mining

In [ ]:
games_df.info()

In [ ]:
games_df.head()

In [ ]:
descriptions = games_df.dropna(subset=['AboutText'])

In [ ]:
descriptions.head()

# Textblob and wordcloud
Credit - https://www.kaggle.com/allunia/eda-en-text-normalization

In the below section I will analyse the importance of a games description in depicting it's genre, and how well the words use in descriptions accurately predict which genre the game is. 

In [ ]:
descriptions = descriptions.drop_duplicates()

In [ ]:
descriptions.info()

In [ ]:
descriptions['DetailedDescrip'] = descriptions['DetailedDescrip'].str.lower()

In [ ]:
descriptions['AboutText'] = descriptions['AboutText'].str.lower()

In [ ]:
descriptions['Genre'].unique()

The following sets out a number of words which would typically be used to describe action games and sees if this accurately reflects words which are used to describe games which are in the action genre.

In [ ]:
#example, calculate the frequency of action- words in descriptions of action games
from __future__ import unicode_literals
from __future__ import division
from textblob import TextBlob

action = descriptions[descriptions.Genre == "Action"]
action_freq = []
for review in action.AboutText:
    review = TextBlob(review)
    num_exciting = review.words.count("exciting")
    num_kill = review.words.count("kill")
    num_steal = review.words.count("steal")
    num_deadly = review.words.count("deadly")
    num_exhilerating = review.words.count("exhilerating")
    num_thrilling = review.words.count("thrilling")
    total_action = num_exciting+num_kill+num_steal+num_deadly+num_exhilerating+num_thrilling
    action_freq.append(total_action)

print(sum(action_freq)/ len(action_freq))

This means that per description that 0.59 of these words are used for action genre games, from here we should see how this compares to other genres, to see whether this is a good score or not.

In [ ]:
import numpy as np
def locate_max(list):
    biggest = np.max(list)
    return biggest, [index for index, element in enumerate(list) 
                      if biggest == element]

In [ ]:
locate_max(action_freq)

In [ ]:
action.AboutText.iloc[73]

In [ ]:
def action_score(descriptions):
    action_freq = []
    for review in descriptions:
        review = TextBlob(review)
        num_exciting = review.words.count("exciting")
        num_kill = review.words.count("kill")
        num_steal = review.words.count("steal")
        num_deadly = review.words.count("deadly")
        num_exhilerating = review.words.count("exhilerating")
        num_thrilling = review.words.count("thrilling")
        total_action = num_exciting+num_kill+num_steal+num_deadly+num_exhilerating+num_thrilling
        action_freq.append(total_action)
    return float(sum(action_freq)/len(action_freq))


In [ ]:
action_score(descriptions.AboutText)

The average score overall is 0.43, significantly less than the score for just action games, showing it is a good signifier for what genre a game is. 

In [ ]:
action_list = []
for genre in descriptions.Genre.unique():
    df_genre = descriptions[descriptions.Genre == genre]
    action = action_score(df_genre.AboutText)
    action_list.append((genre,action))

# sorting from high action to low action   
sorted_action_list = sorted(action_list, key=lambda x: -x[1])

In [ ]:
df_action = pd.DataFrame(sorted_action_list,columns=["Genre","action_score"])

In [ ]:
df_action.head()

In [ ]:
#This performs very very well. 
#Shooter and action are very closely linked, the results are not surprising
import matplotlib.pyplot as plt
plt.rcdefaults()
fig, ax = plt.subplots()
genres = tuple(df_action.Genre.tolist())[:15]
genres = [TextBlob(i) for i in genres]
y_pos = np.arange(len(genres))
performance = np.array(df_action.action_score)[:15]
error = np.random.rand(len(genres))

plt.barh(y_pos, performance, align='center', alpha=0.5)
plt.yticks(y_pos, genres)
plt.title('Game genres by action-score')
 
plt.show()

This visualises what we seen above the only genre that performs better is shooter games, these are closely interlinked meaning that the accuracy is still there. 

In [ ]:
#Let's try the same technique to see what types 
#of words are typically used to describe games which sell well
Highseller = descriptions[descriptions.Global_Sales > np.percentile(descriptions.Global_Sales,90)]

In [ ]:
Highseller.head()

In [ ]:
sports = descriptions[descriptions.Genre == "Sports"]
sports_freq = []
for review in sports.AboutText:
    review = TextBlob(review)
    num_experience = review.words.count("experience")
    num_ball = review.words.count("ball")
    num_realistic = review.words.count("realistic")
    num_latest = review.words.count("latest")
    num_team = review.words.count("team")
    num_pro = review.words.count("pro")
    total_sports = num_experience+num_ball+num_realistic+num_latest+num_team+num_pro
    sports_freq.append(total_sports)

print(sum(sports_freq)/ len(sports_freq))

In [ ]:
def sports_score(descriptions):
    sports_freq = []
    for review in descriptions:
        review = TextBlob(review)
        num_experience = review.words.count("experience")
        num_ball = review.words.count("ball")
        num_realistic = review.words.count("realistic")
        num_latest = review.words.count("latest")
        num_team = review.words.count("team")
        num_pro = review.words.count("pro")
        total_sports = num_experience+num_ball+num_realistic+num_latest+num_team+num_pro
        sports_freq.append(total_sports)
    return float(sum(sports_freq)/len(sports_freq))


In [ ]:
sports_score(descriptions.AboutText)

In [ ]:
sports_list = []
for genre in descriptions.Genre.unique():
    df_genre = descriptions[descriptions.Genre == genre]
    sports = sports_score(df_genre.AboutText)
    sports_list.append((genre,sports))

# sorting from high sweeetness to low sweetness    
sorted_sports_list = sorted(sports_list, key=lambda x: -x[1])

In [ ]:
df_sports = pd.DataFrame(sorted_sports_list,columns=["Genre","sports_score"])

In [ ]:
df_sports.head()

In [ ]:
#This performs very very well. 
#Shooter and action are very closely linked, the results are not 
#surprising
import matplotlib.pyplot as plt
plt.rcdefaults()
fig, ax = plt.subplots()
genres = tuple(df_sports.Genre.tolist())[:15]
genres = [TextBlob(i) for i in genres]
y_pos = np.arange(len(genres))
performance = np.array(df_sports.sports_score)[:15]
error = np.random.rand(len(genres))

plt.barh(y_pos, performance, align='center', alpha=0.5)
plt.yticks(y_pos, genres)
plt.title('Game Genres by sports-score')
 
plt.show()

Furthermore, the description columns have accurately depicted that sports games are sportier than the rest of the columns based on a few words, this is important as it shows that for a game to show what genre it is, the description of that game must contain words which competing games would use themselves.

From here I will analyse the impact of the the words used in reviews in games reviewed badly and reviewed well and see if reviews accurately depict the rating of the game. 

In [ ]:
reviewed = descriptions.dropna()

In [ ]:
reviewed.info()

Creating a dataframe of games which have good reviews, i.e. top 10% user score

In [ ]:
goodreviews = reviewed[reviewed.User_Score > np.percentile(reviewed.User_Score,90)]

In [ ]:
freq = []
for review in goodreviews.Reviews:
    review = TextBlob(review)
    num_best = review.words.count("best")
    num_amazing = review.words.count("amazing")
    num_fantastic = review.words.count("fantastic")
    num_great = review.words.count("great")
    num_brilliant = review.words.count("brilliant")
    num_impressive = review.words.count("impressive")
    total = num_best+num_amazing+num_fantastic+num_great+num_brilliant+num_impressive
    freq.append(total)

print 'The number of good words given in a review of high rated games per review is:' 
print  (sum(freq)/ len(freq))

In [ ]:
freq = []
for review in goodreviews.Reviews:
    review = TextBlob(review)
    num_terrible = review.words.count("terrible")
    num_bad = review.words.count("bad")
    num_worst = review.words.count("worst")
    num_never = review.words.count("never")
    num_regret = review.words.count("regret")
    num_waste = review.words.count("waste")
    total = num_terrible+num_bad+num_worst+num_never+num_regret+num_waste
    freq.append(total)

print 'The number of bad words given in a review of high rated games per review is:' 
print  (sum(freq)/ len(freq))

The results of these are as expected, games with high scores receieve no words which would suggest it is a bad a game and a high rate of words which would suggest it is a great game.

In [ ]:
badreviews = reviewed[reviewed.User_Score < np.percentile(reviewed.User_Score,10)]

In [ ]:
freq = []
for review in badreviews.Reviews:
    review = TextBlob(review)
    num_best = review.words.count("best")
    num_amazing = review.words.count("amazing")
    num_fantastic = review.words.count("fantastic")
    num_great = review.words.count("great")
    num_brilliant = review.words.count("brilliant")
    num_impressive = review.words.count("impressive")
    total = num_best+num_amazing+num_fantastic+num_great+num_brilliant+num_impressive
    freq.append(total)

print 'The number of good words given in a review of bad rated games per review is:' 
print  (sum(freq)/ len(freq))

In [ ]:
freq = []
for review in badreviews.Reviews:
    review = TextBlob(review)
    num_terrible = review.words.count("terrible")
    num_bad = review.words.count("bad")
    num_worst = review.words.count("worst")
    num_never = review.words.count("never")
    num_regret = review.words.count("regret")
    num_waste = review.words.count("waste")
    total = num_terrible+num_bad+num_worst+num_never+num_regret+num_waste
    freq.append(total)

print 'The number of bad words given in a review of bad rated games per review is:' 
print  (sum(freq)/ len(freq))

The first result is surprising, games with low user score have even higher rate of words which are very positive about the game. Suggesting that reviews depend on the user and in order to get an accurate user opinion we must have a large array of reviews and an average score. 
The second result is less so confusing as it shows what we expect in that low scoring games receive a high amount of bad words to describe them. In contrast to the 0 words to describe high rated games. 

From here I will Use wordcloud to analyse what words are typically used to describe games from different genres. 
And games that sell well or really badly, whether words are used to describe high selling games and the same for low selling games. 

In [ ]:
Actiongames  = descriptions[descriptions['Genre'].str.contains('Action')]

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=16                #10 
mpl.rcParams['savefig.dpi']=200             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Actiongames['AboutText']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()


In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Actiongames['DetailedDescrip']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()


In [ ]:
Shootergames = descriptions[descriptions['Genre'].str.contains('Shooter')]

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Shootergames['DetailedDescrip']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()


In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Shootergames['AboutText']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()


In [ ]:
Sportsgames  = descriptions[descriptions['Genre'].str.contains('Sports')]

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Sportsgames['AboutText']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()


In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Sportsgames['DetailedDescrip']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
Highseller = descriptions[descriptions.Global_Sales > np.percentile(descriptions.Global_Sales,90)]

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=300,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Highseller['DetailedDescrip']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Highseller['AboutText']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
Lowsellers = descriptions[descriptions.Global_Sales < np.percentile(descriptions.Global_Sales,10)]

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=200,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Lowsellers['AboutText']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=300,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Lowsellers['DetailedDescrip']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
from subprocess import check_output
from wordcloud import WordCloud, STOPWORDS
mpl.rcParams['font.size']=12                #10 
mpl.rcParams['savefig.dpi']=100             #72 
mpl.rcParams['figure.subplot.bottom']=.1 


stopwords = set(STOPWORDS)

wordcloud = WordCloud(
                          background_color='white',
                          stopwords=stopwords,
                          max_words=300,
                          max_font_size=40, 
                          random_state=42
                         ).generate(str(Lowsellers['Reviews']))

print(wordcloud)
fig = plt.figure(1)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

# Correlation between Columns 
Credit: https://inclass.kaggle.com/ignacioch/predicting-vg-hits-1-million-sales-with-lr-rfc

From here to analyse what factors have the biggest impact on mainly global_sales, we will see which columns are correlated mostly highly. 

In [ ]:
games_df.head()

We will need to drop the review data as it may cause issues with machine learning and correlations due to its mass of NA data and pricefinal will need to be filled with the mean of its column for the sake of accuracy. 

In [ ]:
games_df.drop(['AboutText'] ,inplace =True, axis =1)
games_df.drop(['ShortDescrip'] ,inplace =True, axis =1)
games_df.drop(['DetailedDescrip'] ,inplace =True, axis =1)
games_df.drop(['Reviews'] ,inplace =True, axis =1)
games_df['PriceFinal'].fillna((games_df['PriceFinal'].mean()), inplace=True)

In [ ]:
games_df.head()

In [ ]:
df = games_df.copy()

In [ ]:
df['Platform'] = df['Platform'].astype(object)

Setting the columns up so that they can be changed to keys later for the sake of machine learning. 

In [ ]:
cols = ['Platform', 'Genre', 'Publisher', 'Rating']
for col in cols:
    uniques = df[col].value_counts().keys()
    uniques_dict = {}
    ct = 0
    for i in uniques:
        uniques_dict[i] = ct
        ct += 1

    for k, v in uniques_dict.items():
        df.loc[df[col] == k, col] = v

In [ ]:
df1 = df[['Platform','Genre','Publisher','Year_of_Release','Critic_Score','User_Score','Global_Sales', 'PriceFinal']]
df1 = df1.dropna().reset_index(drop=True)
df1 = df1.astype('float64')

From here we will work from games copied from the original dataframe so we aren't messing it up for future use. 
Also we will disregard all sales apart from Global_Sales as this is the important number from the perspective of the client.

In [ ]:
mask = np.zeros_like(df1.corr())
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(730, 300, sep=20, as_cmap=True, s=85, l=15, n=20) # note: 680, 350/470
with sns.axes_style("white"):
    fig, ax = plt.subplots(1,1, figsize=(15,8))
    ax = sns.heatmap(df1.corr(), mask=mask, vmax=0.2, square=True, annot=True, fmt=".3f", cmap=cmap)

From the above diagram we can see the strongest correlations are:

- Critic scores-to-global sales: We'll take a closer look at this below.

- Year of release-to-platform: This makes sense since new platforms come out periodically.

- User_Score to Critic_Score: This also make sense as they both usually follow the same pattern.

- Price_Final to Platform - This makes sense since new prices come out periodically.

In [ ]:
games_df.head()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12,5))
sns.regplot(x="Critic_Score", y="Global_Sales", data=df1, ci=None, color="#75556c", x_jitter=.02).set(ylim=(0, 17.5))

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12,5))
sns.regplot(x="Critic_Score", y="Global_Sales", data=df1.loc[df1.Year_of_Release >= 2014],
            truncate=True, x_bins=15, color="#75556c").set(ylim=(0,4), xlim=(50, 95))

These diagrams show as the correlation in that as Critic_Score goes up, in general the global_Sales rise with it, only marginally but at the lower ends of sales, it is an important factor to influence global_Sales.

This ties back into a diagram referenced above, that certain genres arelikely to recieve higher scores, this should be considered as it will impact the games sales.

# Machine learning prep

Checking the skewedness of numerical data.
For numerical machine learning this will have to be unskewed for accuracy of prediction. 

In [ ]:
("The skewness of Global_Sales is {}".format(games_df['Global_Sales'].skew()))

In [ ]:
("The skewness of NA_Sales {}".format(games_df['NA_Sales'].skew()))

In [ ]:
("The skewness of EU_Sales {}".format(games_df['EU_Sales'].skew()))

In [ ]:
("The skewness of Other_Sales {}".format(games_df['Other_Sales'].skew()))

In [ ]:
("The skewness of User_Score {}".format(games_df['User_Score'].skew()))

In [ ]:
("The skewness of Critic_Score {}".format(games_df['Critic_Score'].skew()))

As you can see out sales figure are skewed horribly, this needs to be considered when using machine learning, or predicting future values.
Most machine learning techniques assume normally skewed data, as a result of this before we undertake any machine learning we should deskew the data so it increases the accuracy of the results. 
I will attempt to use boxcox as it will most accurately unskew the data

In [ ]:
plt.hist(games_df['Global_Sales'], bins=100)
plt.show()

# Machine learning techniques

In [ ]:
games_df.head()

Carrying on from above we will attempt to see the importance of the name of a game. 

This will be an sklearn machine learning to see how accurately the name of the game depicts the genre of the game. 

In [ ]:
genrelist = ['Action','Adventure','Fighting','Platform','Puzzle','Sports','Racing','Role_Playing','Shooter','Strategy']
subdata = games_df[games_df['Genre'].isin(genrelist)]
subdata = subdata.dropna()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder = label_encoder.fit(subdata['Genre'])
label_encoded_y = label_encoder.transform(subdata['Genre'])
subdata['encoded_genre'] = label_encoded_y
subdata.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    min_df=5, max_features=100, strip_accents='unicode',lowercase =True,
    analyzer='word', token_pattern=r'\w+', use_idf=True, 
    smooth_idf=True, sublinear_tf=True, stop_words = 'english').fit(subdata["Name"])

In [ ]:
features = tfidf.get_feature_names()

In [ ]:
features

In [ ]:
X_tfidf_text = tfidf.transform(subdata["Name"])
subdata_2 = pd.DataFrame(X_tfidf_text.toarray())
subdata = subdata.reset_index()
subdata_2['encoded_genre'] = subdata['encoded_genre']
#Also adding variety for better readibility
subdata_2['Genre'] = subdata['Genre']

In [ ]:
from sklearn.cross_validation import train_test_split
seed = 7

#Split into train and test
test_size = 0.2
y = subdata_2['encoded_genre']
X = subdata_2.drop(['encoded_genre','Genre'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)
# fit model no training data
import xgboost as xgb
clf = xgb.XGBClassifier(max_depth=3, n_estimators=300, learning_rate=0.05)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

This tells us that the features included within the name depict to an accuracy of 50% what genre the game will be. Showing us the importance of a good name, and how it will show the type of game it is, meaning for a games company it will be important for the name to be appropriate. 

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_pred, y_test)
"Accuracy is: %.2f%%" % (accuracy * 100.0)

In [ ]:
games_df.head()

In [ ]:
dfa = games_df
dfa = dfa.copy()
dfa.head()

In [ ]:
dfb = dfa[['Name','Platform','Genre','Publisher','Year_of_Release','Critic_Score','Global_Sales']]
dfb = dfb.dropna().reset_index(drop=True)
df2 = dfb[['Platform','Genre','Publisher','Year_of_Release','Critic_Score','Global_Sales']]
df2['Hit'] = df2['Global_Sales']
df2.drop('Global_Sales', axis=1, inplace=True)

In [ ]:
def hit(sales):
    if sales >= 1:
        return 1
    else:
        return 0

df2['Hit'] = df2['Hit'].apply(lambda x: hit(x))

# Machine learning techniques

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn import svm

For this element of machine learning. We will use SKlearn to see what the biggest factors are in impacting whether a game will be a 'HIT' in the future, meaning that it will sell over 1million copies. 

In [ ]:
df2[:5]

In [ ]:
df_copy = pd.get_dummies(df2)

In [ ]:
df_copy[:5]

In [ ]:
df3 = df_copy
y = df3['Hit'].values
df3 = df3.drop(['Hit'],axis=1)
X = df3.values

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.50, random_state=2)

In [ ]:
radm = RandomForestClassifier(random_state=2).fit(Xtrain, ytrain)
y_val_1 = radm.predict_proba(Xtest)
print("Validation accuracy: ", sum(pd.DataFrame(y_val_1).idxmax(axis=1).values
                                   == ytest)/len(ytest))

In [ ]:
log_reg = LogisticRegression().fit(Xtrain, ytrain)
y_val_2 = log_reg.predict_proba(Xtest)
print("Validation accuracy: ", sum(pd.DataFrame(y_val_2).idxmax(axis=1).values
                                   == ytest)/len(ytest))

In [ ]:
all_predictions = log_reg.predict(Xtest)
print(classification_report(ytest, all_predictions))

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,2.5))
sns.heatmap(confusion_matrix(ytest, all_predictions), annot=True, linewidths=.5, ax=ax, fmt="d").set(xlabel='Predicted Value', ylabel='Expected Value')
#'Training Set Confusion Matrix'

In [ ]:
indices = np.argsort(radm.feature_importances_)[::-1]

# Print the feature ranking
print('Feature ranking (top 10):')

for f in range(10):
    print('%d. feature %d %s (%f)' % (f+1 , indices[f], df3.columns[indices[f]],
                                      radm.feature_importances_[indices[f]]))

As seen above we can see that this puts the following features as things that will make a game sell more than 1 million hits. This is as expected that Critic Score is number one, followed by publisher, genre and Platform. These seem reasonable and we will apply this to the below games which were recently released and under 1m sales. 

In [ ]:
not_hit_copy = df_copy[df_copy['Hit'] == 0]

In [ ]:
df4 = not_hit_copy
y = df4['Hit'].values
df4 = df4.drop(['Hit'],axis=1)
X = df4.values

In [ ]:
df4.head()

In [ ]:
pred = log_reg.predict_proba(X)

In [ ]:
dfb.head()

In [ ]:
dfb = dfb[dfb['Global_Sales'] < 1]

In [ ]:
dfb['Hit_Probability'] = pred[:,1]

In [ ]:
dfb = dfb[dfb['Year_of_Release'] == 2016]
dfb.sort_values(['Hit_Probability'], ascending=[False], inplace=True)
dfb = dfb[['Name', 'Platform', 'Hit_Probability']]

In [ ]:
dfb[:100].reset_index(drop=True)

In [ ]:
dfb[100:-1].reset_index(drop=True)

The results of this are excellent as the games at the top from experience are games which are likely to sell very well and the games are the bottom are very unlikely to sell well, showing the accuracy of the machine learning concepts. 

In [ ]:
gamespred = games_df.copy()

In [ ]:
gamespred.dtypes

# LMfit

We will drop and change the type and unskew columns which could have a negative impact on the accuracy of the upcoming machine learning. 

In [ ]:
gamespred.drop(['NA_Sales'] ,inplace =True, axis =1)
gamespred.drop(['EU_Sales'] ,inplace =True, axis =1)
gamespred.drop(['JP_Sales'] ,inplace =True, axis =1)
gamespred.drop(['Other_Sales'] ,inplace =True, axis =1)
gamespred.drop(['Name'] ,inplace =True, axis =1)

In [ ]:
gamespred['Platform'] = gamespred['Platform'].astype(object)

In [ ]:
gamespred['Global_Sales'] = np.log1p(gamespred['Global_Sales'])

In [ ]:
gamespred['Global_Sales'].skew()

In [ ]:
gamespred['Global_Sales'] = np.log1p(gamespred['Global_Sales'])
gamespred['Global_Sales'].skew()

This is an acceptable level of skewedness, it is now acceptable that the machine learning will be appropriately accurate. 

In [ ]:
gamespred.head()

In [ ]:
gamespred = gamespred.dropna().reset_index(drop=True)

In [ ]:
gamespred.head()

In [ ]:
gamespred.dtypes

In [ ]:
dummy_df = pd.get_dummies(gamespred)

In [ ]:
dummy_df = dummy_df.reset_index()

In [ ]:
dummy_df.rename(columns={'index':'Id'}, inplace=True)

In [ ]:
dummy_df.head()

From here will we use LMfit to see the best fit for Critic score- User-score, Critic_Score-global_sales and USer_Score to Global_Sales

In [ ]:
from numpy import exp 
def gaussian(x, amp, cen, wid): 
    return amp * exp(-(x-cen)**2 / wid)

In [ ]:
import lmfit
from lmfit import Model 
gmodel = Model(gaussian)

gmodel.param_names
gmodel.independent_vars

params = gmodel.make_params(cen=5, amp = 200, wid =1)

In [ ]:
gmodel.set_param_hint('a', value=1.0)
gmodel.set_param_hint('b', value=0.3, min=0, max=1.0)

In [ ]:
y = dummy_df.User_Score
x = dummy_df.Critic_Score

In [ ]:
result = gmodel.fit(y, params, x=x, amp=5,cen=5,wid=1)

In [ ]:
print(result.fit_report())

In [ ]:
plt.plot(x, y,         'bo')
plt.plot(x, result.init_fit, 'k--')
plt.plot(x, result.best_fit, 'r-')
plt.title('correlation between LM fit Critic and user score')
plt.show()

In [ ]:
from numpy import exp 
def gaussian(x, amp, cen, wid): 
    return amp * exp(-(x-cen)**2 / wid)

In [ ]:
import lmfit
from lmfit import Model 
gmodel = Model(gaussian)

gmodel.param_names
gmodel.independent_vars

params = gmodel.make_params(cen=5, amp = 200, wid =1)

In [ ]:
gmodel.set_param_hint('a', value=1.0)
gmodel.set_param_hint('b', value=0.3, min=0, max=1.0)

In [ ]:
y = dummy_df.Global_Sales
x = dummy_df.User_Score

In [ ]:
result = gmodel.fit(y, params, x=x, amp=5,cen=5,wid=1)

In [ ]:
print(result.fit_report())

In [ ]:
plt.plot(x, y,         'bo')
plt.plot(x, result.init_fit, 'k--')
plt.plot(x, result.best_fit, 'r-')
plt.show()

In [ ]:
gmodel.set_param_hint('a', value=1.0)
gmodel.set_param_hint('b', value=0.3, min=0, max=1.0)

In [ ]:
y = dummy_df.Global_Sales
x = dummy_df.Critic_Score

In [ ]:
print(result.fit_report())

In [ ]:
plt.plot(x, y,         'bo')
plt.plot(x, result.init_fit, 'k--')
plt.plot(x, result.best_fit, 'r-')
plt.show()

# Linear Regression

From here will we use linear regression to see which coefficents it will come up with which will most openly impact Global_Sales.

In [ ]:
from sklearn.linear_model import LinearRegression
X = dummy_df.drop('Global_Sales', axis = 1)

lm = LinearRegression()
lm

In [ ]:
lm.fit(X, dummy_df.Global_Sales)

In [ ]:
lm.intercept_

In [ ]:
len(lm.coef_)

In [ ]:
coeffs = pd.DataFrame(zip(X.columns, lm.coef_), columns = ['features', 'estimatedCoefficients'])

In [ ]:
coeffs.sort_values('estimatedCoefficients')

These seem like reasonable coefficients to impact the sales, however I believe that they are skewed by old data and these should be re run to predict more accurate coefficients. 

In [ ]:
lm.predict(X)[0:5]

In [ ]:
plt.scatter(dummy_df.Global_Sales, lm.predict(X))
#plt.ylabel("Predicted sales")
#plt.xlabel("Real sales")
plt.title("sales vs predicted sales")
plt.show()


In [ ]:
lm.score(X,y)

In [ ]:
mseFull = np.mean((dummy_df.Global_Sales - lm.predict(X)) **2)
mseFull
# this is a good mean squared error, showing that this 
#is a very good predictor of future prices. 

This means squared error shows that these parameters are accurate for the linear regression and the score is excellent meaning the coefficiencts provided should be relatively accurate. 

In [ ]:
dummy_df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
X = dummy_df.drop('Critic_Score', axis = 1)

lm = LinearRegression()
lm

In [ ]:
lm.fit(X, dummy_df.Critic_Score)

In [ ]:
lm.intercept_

In [ ]:
len(lm.coef_)

In [ ]:
coeffs = pd.DataFrame(zip(X.columns, lm.coef_), columns = ['features', 'estimatedCoefficients'])

In [ ]:
coeffs.sort_values('estimatedCoefficients')

In [ ]:
lm.predict(X)[0:5]

In [ ]:
plt.scatter(dummy_df.Critic_Score, lm.predict(X))
#plt.ylabel("Predicted sales")
#plt.xlabel("Real sales")
plt.title("Critic_Score vs predicted Critic_Score")
plt.show()


In [ ]:
mseFull = np.mean((dummy_df.Critic_Score - lm.predict(X)) **2)
mseFull
# this is a good mean squared error, showing that this 
#is a very good predictor of future prices. 

Furthermore, the regression model provides us with good coefficients that should impact critic score and inadvertantly increase Global_Sales. The most important coefficients is genre which was suggested in the visulisations earlier in the notebook. 

# Further Models

Now we are going to use regularized linear regression models from the scikit learn module. I'm going to try both l_1(Lasso) and l_2(Ridge) regularization. I'll also define a function that returns the cross-validation rmse error so we can evaluate our models and pick the best tuning par.
Credit: https://www.kaggle.com/apapiu/regularized-linear-models

I will use the Lasso model and the xgBoost to predict what coefficients impact sales most and predict future sales from this. 

In [ ]:
from sklearn.cross_validation import train_test_split
seed = 7

#Split into train and test
test_size = 0.2

X = dummy_df.drop(['Global_Sales'], axis=1)

X_train = X[:X.shape[0]]
X_test = X
y = dummy_df['Global_Sales']

# fit model no training data

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV, ElasticNet, LassoCV, LassoLarsCV
from sklearn.model_selection import cross_val_score

def rmse_cv(model):
    rmse= np.sqrt(-cross_val_score(model, X_train, y, scoring="neg_mean_squared_error", cv = 5))
    return(rmse)

In [ ]:
model_ridge = Ridge()

In [ ]:
alphas = [0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 50, 75]
cv_ridge = [rmse_cv(Ridge(alpha = alpha)).mean() 
            for alpha in alphas]

In [ ]:
cv_ridge = pd.Series(cv_ridge, index = alphas)
cv_ridge.plot(title = "Validation - Just Do It")
#plt.xlabel("alpha")
#plt.ylabel("rmse")

In [ ]:
cv_ridge.min()

In [ ]:
model_lasso = LassoCV(alphas = [1, 0.1, 0.001, 0.0005]).fit(X_train, y)

In [ ]:
rmse_cv(model_lasso).mean()

In [ ]:
coef = pd.Series(model_lasso.coef_, index = X_train.columns)

In [ ]:
("Lasso picked " + str(sum(coef != 0)) + " variables and eliminated the other " +  str(sum(coef == 0)) + " variables")

In [ ]:
imp_coef = pd.concat([coef.sort_values().head(10),
                     coef.sort_values().tail(10)])

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (8.0, 10.0)
imp_coef.plot(kind = "barh")
plt.title("Coefficients in the Lasso Model")

These coefficients seems sort of accurate but again skewed by outdated data again. 

In [ ]:
#let's look at the residuals as well:
matplotlib.rcParams['figure.figsize'] = (6.0, 6.0)

preds = pd.DataFrame({"preds":model_lasso.predict(X_train), "true":y})
preds["residuals"] = preds["true"] - preds["preds"]
preds.plot(x = "preds", y = "residuals",kind = "scatter")

In [ ]:
import xgboost as xgb

In [ ]:
dtrain = xgb.DMatrix(X_train, label = y)
dtest = xgb.DMatrix(X_test)

params = {"max_depth":2, "eta":0.1}
model = xgb.cv(params, dtrain,  num_boost_round=500, early_stopping_rounds=100)

In [ ]:
model.loc[30:,["test-rmse-mean", "train-rmse-mean"]].plot()

In [ ]:
model_xgb = xgb.XGBRegressor(n_estimators=360, max_depth=2, learning_rate=0.1) #the params were tuned using xgb.cv
print(model_xgb.fit(X_train, y))

In [ ]:
rmse_cv(model_xgb).mean()

In [ ]:
xgb_preds = np.expm1(model_xgb.predict(X_test))
lasso_preds = np.expm1(model_lasso.predict(X_test))

In [ ]:
predictions = pd.DataFrame({"xgb":xgb_preds, "lasso":lasso_preds})
predictions.plot(x = "xgb", y = "lasso", kind = "scatter")

In [ ]:
preds = xgb_preds

In [ ]:
solution1 = pd.DataFrame({"id":dummy_df.Id, "Global_Sales":dummy_df.Global_Sales, "Pred_Sales":preds })

In [ ]:
solution1

In [ ]:
preds = lasso_preds

In [ ]:
solution2 = pd.DataFrame({"id":dummy_df.Id, "Global_Sales":dummy_df.Global_Sales, "Pred_Sales":preds })

In [ ]:
solution2

In [ ]:
preds = 0.7*lasso_preds + 0.3*xgb_preds

In [ ]:
solution = pd.DataFrame({"id":dummy_df.Id, "Global_Sales":dummy_df.Global_Sales, "Pred_Sales":preds })

In [ ]:
solution

In [ ]:
preds = 0.85*lasso_preds + 0.15*xgb_preds

In [ ]:
solution3 = pd.DataFrame({"id":dummy_df.Id, "Global_Sales":dummy_df.Global_Sales, "Pred_Sales":preds })

In [ ]:
solution3

The most accurate model is a mixture of the xgb model and the lasso model gives an almost 10% accurate depication of sales, meaning it could predict future sales given the parameters with good accuracy. 

The results received from this are satisfactory, however, as done above I believe that it may be more accurate at predicting useful coefficients for the client if the dataset focuses on the relevant years, I will do so below with the Lasso model and Linear Regression. 

In [ ]:
dummy_df.head()

In [ ]:
relevant_games2 = dummy_df[dummy_df.Year_of_Release.isin(relevant_years)]

In [ ]:
relevant_games2.head()

In [ ]:
from sklearn.linear_model import LinearRegression
X = relevant_games2.drop('Global_Sales', axis = 1)

lm = LinearRegression()
lm

In [ ]:
lm.fit(X, relevant_games2.Global_Sales)

In [ ]:
lm.intercept_

In [ ]:
len(lm.coef_)

In [ ]:
coeffs = pd.DataFrame(zip(X.columns, lm.coef_), columns = ['features', 'estimatedCoefficients'])

In [ ]:
coeffs.sort_values('estimatedCoefficients', ascending =True)

The linear regression parameters given are surprising, however seeing there price vs predicted price they are somewhat accurate. 
Past publisher it recommends: 
Ps4, shooter, adventure and also recommends smaller publisher showing good things for the small games CEO. 

In [ ]:
lm.predict(X)[0:5]

In [ ]:
plt.scatter(relevant_games2.Global_Sales, lm.predict(X))
#plt.ylabel("Predicted sales")
#plt.xlabel("Real sales")
plt.title("Price vs predicted price")
plt.show()


In [ ]:
mseFull = np.mean((relevant_games2.Global_Sales - lm.predict(X)) **2)
mseFull
# this is a good mean squared error, showing that this 
#is a very good predictor of future prices. 

In [ ]:
from sklearn.cross_validation import train_test_split
seed = 7

#Split into train and test
test_size = 0.2

X = relevant_games2.drop(['Global_Sales'], axis=1)

X_train = X[:X.shape[0]]
X_test = X
y = relevant_games2['Global_Sales']

# fit model no training data

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV, ElasticNet, LassoCV, LassoLarsCV
from sklearn.model_selection import cross_val_score

def rmse_cv(model):
    rmse= np.sqrt(-cross_val_score(model, X_train, y, scoring="neg_mean_squared_error", cv = 5))
    return(rmse)

In [ ]:
model_ridge = Ridge()

In [ ]:
alphas = [0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 50, 75]
cv_ridge = [rmse_cv(Ridge(alpha = alpha)).mean() 
            for alpha in alphas]

In [ ]:
cv_ridge = pd.Series(cv_ridge, index = alphas)
cv_ridge.plot(title = "Validation - Just Do It")
#plt.xlabel("alpha")
#plt.ylabel("rmse")

In [ ]:
cv_ridge.min()

In [ ]:
model_lasso = LassoCV(alphas = [1, 0.1, 0.001, 0.0005]).fit(X_train, y)

In [ ]:
rmse_cv(model_lasso).mean()

In [ ]:
coef = pd.Series(model_lasso.coef_, index = X_train.columns)

In [ ]:
("Lasso picked " + str(sum(coef != 0)) + " variables and eliminated the other " +  str(sum(coef == 0)) + " variables")

In [ ]:
imp_coef = pd.concat([coef.sort_values().head(15),
                     coef.sort_values().tail(15)])

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (8.0, 10.0)
imp_coef.plot(kind = "barh")
plt.title("Coefficients in the Lasso Model")

This is an excellent list of coeffiencts as it tells us things that we previously knew, that Action, adventure and shooters games are likely to sell well. Larger publishers are also likely to sell well. But past this Critic Score has an impact, and PS4 is the way to go in terms of platform, recommending against PC releases and Misc genre. 